In [ ]:
%pip install -r requirements.txt

In [ ]:
"""
Handle imports & some utility functions + typing
"""

import os
import re
import folium
from folium.plugins import Draw
from pathlib import Path
from typing import Any, List, Optional, Sequence, Tuple, Union

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import odc.geo.xr  # noqa: F401
import rasterio
import tabulate
import xarray as xr
from dotenv import load_dotenv
from IPython.display import HTML, display
from odc.stac import load as odc_load
from pystac_client import Client
from rasterio import features
from shapely.geometry import shape

DateType = Union[str, Tuple[str, str], List[str]]
BBoxType = Union[List[float], Tuple[float, float, float, float]]
ArrayLike = Union[xr.DataArray, np.ndarray]

_TABLE_CSS = """
<style>
.qf-wrap { margin: 8px 0 18px; font-family: ui-sans-serif, system-ui, sans-serif; color: inherit; }
.qf-wrap h3 { margin: 0 0 4px; font-size: 1.15rem; color: inherit; }
.qf-wrap .sub { margin: 0 0 10px; opacity: 0.75; font-size: 0.92rem; color: inherit; }
.qf-table {
  border-collapse: collapse; width: auto; max-width: 100%; font-size: 13px;
  background: transparent; color: inherit;
}
.qf-table th {
  background: transparent; color: inherit; padding: 8px 12px; text-align: left;
  border: none; border-bottom: 2px solid rgba(128,128,128,0.45); white-space: nowrap;
  font-weight: 600;
}
.qf-table td {
  background: transparent !important; color: inherit; padding: 8px 12px;
  border: none; border-bottom: 1px solid rgba(128,128,128,0.25); vertical-align: top;
}
.qf-badge {
  display: inline-block; padding: 2px 10px; border-radius: 999px;
  font-weight: 600; font-size: 12px; letter-spacing: 0.02em;
}
.qf-badge-ok { background: #166534; color: #dcfce7; }
.qf-badge-no { background: #991b1b; color: #fee2e2; }
</style>
"""


def parse_bbox(value: str) -> List[float]:
    """Parse BBOX=west,south,east,north from .env into four floats."""
    parts = [float(x.strip()) for x in value.split(",")]
    if len(parts) != 4:
        raise ValueError(f"BBOX must have 4 comma-separated floats, got: {value!r}")
    return parts


def _normalize_datetime(datetime: DateType) -> str:
    """Normalize datetime to a string format."""
    if isinstance(datetime, str):
        return datetime
    if isinstance(datetime, (list, tuple)) and len(datetime) == 2:
        return f"{datetime[0]}/{datetime[1]}"
    raise TypeError("datetime must be a string or a (start, end) pair")


def show_section(title: str, subtitle: str | None = None) -> None:
    """Render an HTML section header."""
    sub = f'<p class="sub">{subtitle}</p>' if subtitle else ""
    display(HTML(f'{_TABLE_CSS}<div class="qf-wrap"><h3>{title}</h3>{sub}</div>'))


def format_crs(crs: Any) -> str:
    """Compact CRS label, e.g. EPSG:3857."""
    if crs is None:
        return "n/a"
    epsg = getattr(crs, "to_epsg", lambda: None)()
    if epsg is None:
        epsg = getattr(crs, "epsg", None)
    if epsg is not None:
        return f"EPSG:{int(epsg)}"
    text = str(crs)
    match = re.search(r'ID\["EPSG",\s*(\d+)\]', text) or re.search(r"EPSG[:\s]*(\d+)", text, re.I)
    if match:
        return f"EPSG:{match.group(1)}"
    return text if len(text) <= 48 else text[:45] + "..."


def display_table(
    data: Sequence[Sequence[Any]],
    headers: Sequence[str],
    *,
    title: str | None = None,
) -> None:
    """Render a styled HTML table in the notebook."""
    # unsafehtml keeps status badges as real HTML (not escaped text)
    table_html = tabulate.tabulate(data, headers=list(headers), tablefmt="unsafehtml")
    table_html = table_html.replace("<table>", '<table class="qf-table">', 1)
    heading = f"<div style='font-weight:600;margin-bottom:6px'>{title}</div>" if title else ""
    display(HTML(f'{_TABLE_CSS}<div class="qf-wrap">{heading}{table_html}</div>'))


def status_badge(ok: bool, yes: str = "DETECTED", no: str = "NOT DETECTED") -> str:
    cls = "qf-badge-ok" if ok else "qf-badge-no"
    label = yes if ok else no
    return f'<span class="qf-badge {cls}">{label}</span>'


In [ ]:
"""
Data Acquisition
"""

# Element84 sentinel-2-l2a asset names <-> Sentinel-2 band IDs from Copernicus
S2_BANDS = ("blue", "green", "red", "nir", "nir08", "swir16", "swir22", "scl")
BAND_ALIASES = {
    "B02": "blue",
    "B03": "green",
    "B04": "red",
    "B08": "nir",
    "B8A": "nir08",
    "B11": "swir16",
    "B12": "swir22",
}

def get_data(
    bbox: BBoxType,
    datetime: DateType,
    *,
    bands: Sequence[str] = S2_BANDS,
    crs: str = "EPSG:3857",
    resolution: int = 10,
    groupby: str = "solar_day",
    chunks: Optional[dict] = None,
    collection: str = "sentinel-2-l2a",
) -> xr.Dataset:
    """Load Sentinel-2 L2A bands for a bbox/datetime from Element84 Earth Search."""
    if not isinstance(bbox, (list, tuple)) or len(bbox) != 4:
        raise ValueError("bbox must be a sequence of 4 floats: [west, south, east, north]")

    datetime_str = _normalize_datetime(datetime)
    client = Client.open("https://earth-search.aws.element84.com/v1")
    search = client.search(collections=[collection], bbox=bbox, datetime=datetime_str)
    items = list(search.items())
    if not items:
        raise ValueError(f"No STAC items found for bbox={bbox!r}, datetime={datetime_str!r}")

    load_kwargs = {
        "bands": list(bands),
        "bbox": bbox,
        "crs": crs,
        "resolution": resolution,
        "groupby": groupby,
    }
    if chunks is not None:
        load_kwargs["chunks"] = chunks

    return odc_load(items, **load_kwargs)

In [ ]:
"""
Vectorized QuickFire V1.0.0 (Pierre Markuse) for xarray / numpy arrays.
Expects Element84 band names and reflectance in ~0-1 (divide DN by 10_000).
"""

HS_THRESHOLDS = (2.0, 1.5, 1.25, 1.0)


def to_reflectance(ds: xr.Dataset, scale: float = 10_000.0) -> xr.Dataset:
    """Convert Sentinel-2 L2A digital numbers to reflectance."""
    out = ds.copy()
    for name in ("blue", "green", "red", "nir", "nir08", "swir16", "swir22"):
        if name in out:
            out[name] = out[name].astype("float32") / scale
    return out


def stretch(val: ArrayLike, vmin: float, vmax: float) -> ArrayLike:
    return (val - vmin) / (vmax - vmin)


def sat_enh(r: ArrayLike, g: ArrayLike, b: ArrayLike, s: float):
    avg = (r + g + b) / 3.0
    return avg * (1.0 - s) + r * s, avg * (1.0 - s) + g * s, avg * (1.0 - s) + b * s


def layer_blend(lay1, lay2, lay3, op1: float, op2: float, op3: float):
    return tuple(
        a / 100.0 * op1 + b / 100.0 * op2 + c / 100.0 * op3
        for a, b, c in zip(lay1, lay2, lay3)
    )


def is_cloud(blue: ArrayLike, green: ArrayLike, red: ArrayLike) -> ArrayLike:
    """Cloud heuristic from the Copernicus Browser QuickFire evalscript."""
    ngdr = (green - red) / (green + red)
    b_ratio = (green - 0.175) / (0.39 - 0.175)
    return (b_ratio > 1.0) | ((b_ratio > 0.0) & (ngdr > 0.0))


def hotspot_level(
    swir16: ArrayLike,
    swir22: ArrayLike,
    *,
    hs_sensitivity: float = 1.0,
    thresholds: Tuple[float, ...] = HS_THRESHOLDS,
) -> ArrayLike:
    """Return 0 (no hotspot) … 4 (strongest) from SWIR sum thresholds."""
    strength = swir22 + swir16
    if isinstance(swir16, xr.DataArray):
        level = xr.zeros_like(swir16, dtype="uint8")
        for i, thr in enumerate(reversed(thresholds), start=1):
            level = xr.where(strength > (thr / hs_sensitivity), np.uint8(i), level)
        return level.rename("hotspot_level")
    level = np.zeros(np.asarray(strength).shape, dtype="uint8")
    for i, thr in enumerate(reversed(thresholds), start=1):
        level = np.where(strength > (thr / hs_sensitivity), i, level)
    return level


def quickfire(
    ds: xr.Dataset,
    *,
    style: int = 1,
    hotspot: bool = True,
    cloud_avoidance: bool = True,
    show_burnscars: bool = False,
    hs_sensitivity: float = 1.0,
    boost: float = 1.2,
    avoidance_helper: float = 0.8,
    water_highlight: bool = False,
) -> Tuple[xr.DataArray, xr.DataArray]:
    """Apply QuickFire visualization.

    Returns
    -------
    rgb : DataArray
        Float RGB image with dims (band, y, x), values roughly in 0–1+.
    hotspot_mask : DataArray
        Uint8 hotspot severity 0-4 (4 = strongest). Cloud-masked when enabled.
    """
    b02, b03, b04 = ds["blue"], ds["green"], ds["red"]
    b08, b8a = ds["nir"], ds["nir08"]
    b11, b12 = ds["swir16"], ds["swir22"]

    offset = -0.007
    saturation = 1.10
    brightness = 1.40
    s_min, s_max = 0.15, 0.99
    burnscar_threshold = -0.25
    burnscar_strength = 0.3
    water_boost = 2.0
    ndvi_threshold = 0.05
    ndwi_threshold = 0.0
    water_helper = 0.1
    manual_correction = (0.04, 0.00, -0.05)

    ndwi = (b03 - b08) / (b03 + b08)
    ndvi = (b08 - b04) / (b08 + b04)
    nbr = (b08 - b12) / (b08 + b12)

    # Guard against tiny negatives before sqrt (offset can push dark pixels < 0).
    def _sqrt_safe(x):
        return np.sqrt(np.maximum(x, 0.0))

    natural_cc = (
        _sqrt_safe(brightness * b04 + offset),
        _sqrt_safe(brightness * b03 + offset),
        _sqrt_safe(brightness * b02 + offset),
    )
    urban = (
        _sqrt_safe(brightness * b12 * 1.2 + offset),
        _sqrt_safe(brightness * b11 * 1.4 + offset),
        _sqrt_safe(brightness * b04 + offset),
    )
    swir = (
        _sqrt_safe(brightness * b12 + offset),
        _sqrt_safe(brightness * b8a + offset),
        _sqrt_safe(brightness * b04 + offset),
    )

    viz_r, viz_g, viz_b = layer_blend(urban, swir, natural_cc, 10, 10, 90)

    if water_highlight:
        water = (ndvi < ndvi_threshold) & (ndwi > ndwi_threshold) & (b04 < water_helper)
        viz_g = xr.where(water, viz_g * 1.2 * water_boost + 0.1, viz_g)
        viz_b = xr.where(water, viz_b * 1.5 * water_boost + 0.2, viz_b)

    viz_r, viz_g, viz_b = sat_enh(viz_r, viz_g, viz_b, saturation)
    viz_r = stretch(viz_r, s_min, s_max) + manual_correction[0]
    viz_g = stretch(viz_g, s_min, s_max) + manual_correction[1]
    viz_b = stretch(viz_b, s_min, s_max) + manual_correction[2]

    if show_burnscars:
        scar = nbr < burnscar_threshold
        viz_r = xr.where(scar, viz_r + burnscar_strength, viz_r)
        viz_g = xr.where(scar, viz_g + burnscar_strength, viz_g)

    levels = hotspot_level(b11, b12, hs_sensitivity=hs_sensitivity)
    eligible = ~is_cloud(b02, b03, b04) & (b02 < avoidance_helper) if cloud_avoidance else True
    levels = xr.where(eligible, levels, 0) if hotspot else xr.zeros_like(b02, dtype="uint8")

    if hotspot:
        # style 1: progressive SWIR boost (matches evalscript case 1)
        if style == 1:
            for thr_i, g_boost in ((4, 0.50), (3, 0.20), (2, 0.10), (1, 0.00)):
                m = levels == thr_i
                viz_r = xr.where(m, boost * 0.50 * b12 + viz_r, viz_r)
                viz_g = xr.where(m, boost * g_boost * b11 + viz_g, viz_g)
        elif style == 2:
            m = levels >= 1
            viz_r, viz_g, viz_b = xr.where(m, 1, viz_r), xr.where(m, 0, viz_g), xr.where(m, 0, viz_b)
        elif style == 3:
            m = levels >= 1
            viz_r, viz_g, viz_b = xr.where(m, 1, viz_r), xr.where(m, 1, viz_g), xr.where(m, 0, viz_b)
        elif style == 4:
            m = levels >= 1
            viz_r = xr.where(m, viz_r + 0.2, viz_r)
            viz_g = xr.where(m, viz_g - 0.2, viz_g)
            viz_b = xr.where(m, viz_b - 0.2, viz_b)

    rgb = xr.concat([viz_r, viz_g, viz_b], dim="band").assign_coords(band=["r", "g", "b"])
    rgb = rgb.clip(0, 1)
    levels = levels.astype("uint8").rename("hotspot_level")
    # xr.where can drop odc spatial metadata; reattach CRS from an input band.
    if getattr(b02.odc, "crs", None) is not None:
        rgb = rgb.odc.assign_crs(b02.odc.crs)
        levels = levels.odc.assign_crs(b02.odc.crs)
    return rgb, levels


def hotspots_to_geodataframe(hotspot_mask: xr.DataArray, min_level: int = 1, crs=None):
    """Polygonize hotspot pixels into a GeoDataFrame."""

    mask = hotspot_mask.squeeze(drop=True)
    if "time" in mask.dims:
        raise ValueError("Pass a single time slice to hotspots_to_geodataframe")

    values = np.asarray(mask.values)
    binary = (values >= min_level).astype("uint8")
    transform = mask.odc.transform
    if crs is None:
        crs = mask.odc.crs
    if crs is None and "spatial_ref" in mask.coords:
        crs = int(mask.spatial_ref.item())

    records = []
    for geom, val in features.shapes(values.astype("uint8"), mask=binary.astype(bool), transform=transform):
        if val >= min_level:
            records.append({"hs_level": int(val), "geometry": shape(geom)})

    if not records:
        return gpd.GeoDataFrame(columns=["hs_level", "geometry"], geometry="geometry", crs=crs)

    return gpd.GeoDataFrame(records, crs=crs)

In [ ]:
"""
Main Workflow
"""

# Load variables from .env (see example.env)
load_dotenv()

bbox_raw = os.getenv("BBOX")
datetime = os.getenv("DATETIME")
out_dir = Path(os.getenv("OUT_DIR", "outputs"))

if not bbox_raw or not datetime:
    raise ValueError("Missing BBOX or DATETIME in .env - copy example.env to .env and set values")

bbox = parse_bbox(bbox_raw)
out_dir.mkdir(exist_ok=True)

show_section("Run configuration", "Values loaded from .env")
display_table(
    [[f"{bbox}", datetime, str(out_dir)]],
    ["bbox (west, south, east, north)", "datetime", "out_dir"],
)

# 1) Load STAC COGs for the bands QuickFire needs
data = get_data(bbox, datetime, bands=S2_BANDS, resolution=20)

# 2) Reflectance scale + pick the clearest solar day
refl = to_reflectance(data)
cloud_frac = is_cloud(refl["blue"], refl["green"], refl["red"]).mean(dim=("y", "x"))
t_idx = int(cloud_frac.argmin().item())
scene = refl.isel(time=t_idx)
scene_time = np.datetime_as_string(scene.time.values, unit="s")

show_section("Scene selection", "Clearest solar-day slice by QuickFire cloud heuristic")
display_table(
    [[
        t_idx,
        scene_time,
        f"{float(cloud_frac.isel(time=t_idx)):.3%}",
        f"{int(scene.sizes.get('y', 0))} x {int(scene.sizes.get('x', 0))}",
        format_crs(scene.odc.crs),
        "20 m",
    ]],
    ["time_index", "scene_time (UTC)", "cloud_frac", "shape (y x)", "crs", "resolution"],
)

# 3) Run QuickFire
rgb, hotspot_mask = quickfire(scene, style=1, hotspot=True, cloud_avoidance=True)

# Hotspot level histogram for the selected scene
levels = np.asarray(hotspot_mask.values).squeeze()
level_rows = []
for level in range(5):
    count = int((levels == level).sum())
    pct = (count / levels.size * 100.0) if levels.size else 0.0
    level_rows.append([level, count, f"{pct:.4f}%"])

show_section("Hotspot level distribution", "0 = none, 4 = strongest SWIR hotspot")
display_table(level_rows, ["hs_level", "pixels", "percent_of_scene"])

# 4) Visualize RGB + hotspot overlay
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
rgb.plot.imshow(ax=axes[0], rgb="band")
axes[0].set_title("QuickFire RGB")
hotspot_mask.plot.imshow(ax=axes[1], cmap="hot", vmin=0, vmax=4)
axes[1].set_title("Hotspot level (0-4)")
fig.suptitle(f"Old Trails Fire AOI — {scene_time}", y=1.02)
plt.tight_layout()
plt.show()

# 5) Export hotspot polygons as shapefile + GeoJSON (WGS84)
gdf = hotspots_to_geodataframe(hotspot_mask, min_level=1, crs=scene.odc.crs)
shapefile_dir = out_dir / "shapefile"
geojson_dir = out_dir / "geojson"
shapefile_dir.mkdir(exist_ok=True)
geojson_dir.mkdir(exist_ok=True)

shapefile = shapefile_dir / f"hotspots_{datetime}.shp"
geojson_file = geojson_dir / f"hotspots_{datetime}.geojson"

export_rows = []
preview = None
level_summary = None
if len(gdf):
    gdf_wgs84 = gdf.to_crs("EPSG:4326")
    gdf_wgs84.to_file(shapefile)
    gdf_wgs84.to_file(geojson_file, driver="GeoJSON")
    export_rows.extend(
        [
            ["shapefile", len(gdf_wgs84), str(shapefile.resolve())],
            ["geojson", len(gdf_wgs84), str(geojson_file.resolve())],
        ]
    )

    preview = gdf_wgs84.copy()
    # Centroids in native CRS, then convert to WGS84 for display
    centroids_wgs84 = gpd.GeoSeries(gdf.geometry.centroid, crs=gdf.crs).to_crs("EPSG:4326")
    preview["lon"] = centroids_wgs84.x.round(5).to_numpy()
    preview["lat"] = centroids_wgs84.y.round(5).to_numpy()
    level_summary = (
        preview.groupby("hs_level").size().rename("polygons").reset_index().sort_values("hs_level")
    )

# 6-7) Save SCL + hotspot mask as GeoTIFFs
def write_geotiff(path, array, *, transform, crs, dtype="uint8"):
    data2d = np.asarray(array).squeeze()
    if data2d.ndim != 2:
        raise ValueError(f"Expected 2D array for GeoTIFF, got shape {data2d.shape}")
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    profile = {
        "driver": "GTiff",
        "height": data2d.shape[0],
        "width": data2d.shape[1],
        "count": 1,
        "dtype": dtype,
        "crs": crs,
        "transform": transform,
        "compress": "lzw",
    }
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(data2d.astype(dtype, copy=False), 1)
    return path


transform = scene.odc.transform
crs = scene.odc.crs

scl_tiff = write_geotiff(
    out_dir / "scl" / f"scl_{datetime}.tif",
    scene["scl"].values,
    transform=transform,
    crs=crs,
)
hotspot_tiff = write_geotiff(
    out_dir / "hotspot_mask" / f"hotspot_mask_{datetime}.tif",
    hotspot_mask.values,
    transform=transform,
    crs=crs,
)
export_rows.extend(
    [
        ["scl geotiff", 1, str(scl_tiff.resolve())],
        ["hotspot mask geotiff", 1, str(hotspot_tiff.resolve())],
    ]
)

show_section("Exports", "Written artifacts for this run")
display_table(export_rows, ["product", "count", "path"])

if level_summary is not None and len(level_summary):
    show_section("Polygon summary by hotspot level")
    display_table(level_summary.values.tolist(), ["hs_level", "polygons"])

    show_section("Hotspot polygon preview", "Centroid lon/lat (WGS84), first 10 features")
    display_table(
        preview[["hs_level", "lon", "lat"]].head(10).values.tolist(),
        ["hs_level", "lon", "lat"],
    )
else:
    show_section("Polygon summary", "No hotspot pixels detected for this scene/AOI")

# 8) Threshold the hotspot mask and summarize detection
threshold = 2
fire_pixels = levels >= threshold
fire_pixel_count = int(fire_pixels.sum())
total_pixels = int(levels.size)
detected_percentage = (fire_pixel_count / total_pixels * 100.0) if total_pixels else 0.0
fire_detected = fire_pixel_count > 0

show_section(
    "Fire detection summary",
    f"Active-fire threshold: hotspot level &gt;= {threshold}",
)
display_table(
    [[
        status_badge(fire_detected),
        fire_pixel_count,
        total_pixels,
        f"{detected_percentage:.4f}%",
        len(gdf) if len(gdf) else 0,
    ]],
    ["status", "fire_pixels", "scene_pixels", "percent_of_scene", "polygons_exported"],
)


In [ ]:
"""
Visualize the fire detection results in a simple map
"""

show_section("Map preview", "Hotspot polygons over OpenStreetMap")

if not geojson_file.exists() or len(gdf) == 0:
    display(HTML("<p class='qf-wrap sub'>No hotspot GeoJSON available to map.</p>"))
else:
    # Folium needs a str path / GeoJSON dict - pathlib.Path is not accepted.
    bbox_center = ((bbox[1] + bbox[3]) / 2, (bbox[0] + bbox[2]) / 2)
    m = folium.Map(location=bbox_center, zoom_start=10,
        tiles="CartoDB dark_matter",
        attr='&copy; <a href="https://openstreetmap.org">OpenStreetMap</a> contributors &copy; <a href="https://carto.com">CARTO</a>')

    level_colors = {
        1: "#fbbf24",
        2: "#f97316",
        3: "#ef4444",
        4: "#7f1d1d",
    }

    def style_feature(feature):
        level = int(feature["properties"].get("hs_level", 1))
        color = level_colors.get(level, "#f59e0b")
        return {
            "fillColor": color,
            "color": color,
            "weight": 1,
            "fillOpacity": 0.55,
        }

    folium.GeoJson(
        data=str(geojson_file),
        name="hotspots",
        style_function=style_feature,
        tooltip=folium.GeoJsonTooltip(fields=["hs_level"], aliases=["Hotspot level"]),
    ).add_to(m)

    # Fit to AOI bbox [[south, west], [north, east]]
       


    # 1. Project a temporary copy to standard Lat/Lon (WGS84)
    gdf_wgs84 = gdf.to_crs(epsg=4326)

    # 2. Get the correct lat/lon bounds [xmin, ymin, xmax, ymax]
    xmin, ymin, xmax, ymax = gdf_wgs84.total_bounds

    # 3. Calculate center and dimensions in degrees
    center_x = (xmin + xmax) / 2
    center_y = (ymin + ymax) / 2
    width = xmax - xmin
    height = ymax - ymin

    # 4. Shrink by half to zoom twice as close
    new_xmin = center_x - (width / 4)
    new_xmax = center_x + (width / 4)
    new_ymin = center_y - (height / 4)
    new_ymax = center_y + (height / 4)

    # 5. Fit the map securely to these tight lat/lon bounds
    m.fit_bounds([[new_ymin, new_xmin], [new_ymax, new_xmax]])
    
    folium.LayerControl().add_to(m)

    from folium.plugins import Draw

    Draw(
        export=True,
        filename="bbox.geojson",
        draw_options={'rectangle': True, 'polyline': False, 'polygon': False, 'circle': False, 'marker': False, 'circlemarker': False}
    ).add_to(m)


    # Always write a standalone HTML map (works even when the notebook is untrusted).
    map_dir = out_dir / "maps"
    map_dir.mkdir(exist_ok=True)
    map_html = map_dir / f"hotspots_{datetime}.html"
    m.save(str(map_html))

    display(
        HTML(
            f"""
            <div class="qf-wrap">
              <p class="sub">
                Inline Folium maps require a <b>trusted</b> notebook in Cursor/VS Code:
                Command Palette -> <code>Notebook: Trust</code> (or the Trust banner),
                then re-run this cell.
              </p>
              <p>
                Saved map:
                <a href="{map_html.resolve().as_uri()}" target="_blank" rel="noopener">{map_html.resolve()}</a>
              </p>
            </div>
            """
        )
    )

    # Inline render (shows the trust message if the notebook is still untrusted).
    display(m)
